# 01 · Bronze ingest
Loads the raw IBM AML files from the `raw_files` volume into bronze Delta tables, unchanged except for column names and audit columns.

In [0]:
# List the raw files in the bronze volume, with their size in MB
files = dbutils.fs.ls("/Volumes/workspace/aml_bronze/raw_files/")

for f in files:
    print(f.name, round(f.size / 1_000_000, 1), "MB")

In [0]:
from pyspark.sql import functions as F

VOLUME = "/Volumes/workspace/aml_bronze/raw_files"
BRONZE = "workspace.aml_bronze"

In [0]:
raw_trans = (
    spark.read
    .option("header", True)
    .csv(f"{VOLUME}/HI-Small_Trans.csv")
)

print(raw_trans.columns)

In [0]:
trans_columns = [
    "timestamp", "from_bank", "from_account", "to_bank", "to_account",
    "amount_received", "receiving_currency", "amount_paid",
    "payment_currency", "payment_format", "is_laundering",
]

bronze_trans = (
    raw_trans
    .select("*", "_metadata.file_path")
    .toDF(*trans_columns, "_source_file")
    .withColumn("_ingested_at", F.current_timestamp())
)

bronze_trans.printSchema()

In [0]:
(
    bronze_trans.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{BRONZE}.transactions")
)

In [0]:
raw_accounts = (
    spark.read
    .option("header", True)
    .csv(f"{VOLUME}/HI-Small_accounts.csv")
)

print(raw_accounts.columns)

In [0]:
account_columns = [
    "bank_name", "bank_id", "account_number", "entity_id", "entity_name",
]

bronze_accounts = (
    raw_accounts
    .select("*", "_metadata.file_path")
    .toDF(*account_columns, "_source_file")
    .withColumn("_ingested_at", F.current_timestamp())
)

(
    bronze_accounts.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{BRONZE}.accounts")
)